In [12]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

TABLE = "data/historic_airphoto_index/main_index_blank.gpkg"
OUTPUT_DIR = Path("data/historic_airphoto_index/subsets")

gdf = gpd.read_file(TABLE)

## Prepare the Subset

In [2]:
# CONFIGURATION CELL - EDIT HERE

# specify the county
COUNTY = "Carver"

### Examine available years for a county

In [3]:
mask = gdf["county"] == COUNTY

subset = gdf.loc[mask]

for year in sorted(subset["year"].dropna().unique()):
    print(year)

1937
1938
1940
1947
1951
1953
1955
1956
1957
1960
1962
1963
1964
1967
1969
1971


### Choose Years and Manually Populate Scale Info
Use scale information from PDFs found here: 
https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery

This notebook assumes you're going to choose a set of years that all have the same scale, for efficiency. 

In [4]:
# CONFIGURATION CELL - EDIT HERE

# Choose years for processing
# These should all have the same scale
YEARS = [1937, 1938, 1940, 1951, 1957, 1963]

SCALE = 20000

In [6]:
mask = (
    (gdf["county"] == COUNTY) &
    (gdf["year"].isin(YEARS))
)

subset_gdf = gdf.loc[mask].copy()

print(f"Matching records: {len(subset_gdf):,}")

Matching records: 871


In [7]:
missing_scale = subset_gdf["scale"].isna()

subset_gdf.loc[missing_scale, "scale"] = SCALE

print(f"Scales populated: {missing_scale.sum():,}")
print(subset_gdf["scale"].value_counts(dropna=False))

Scales populated: 871
scale
20000    871
Name: count, dtype: int64


In [ ]:
# check for unexpected scale values
assert subset_gdf["scale"].eq(SCALE).all(), (
    "Some records have a different scale than expected!"
)

In [14]:
subset_gdf

,cty_year_id,status,county,year,photo_id,h_extent,gcp_count,confidence,scale,dpi_x,width_px,worldfile_json,url,flight_code,source_id,date_download,date_georef,geometry
13511,carver_1937_wm-26-2368,None,Carver,1937,wm-26-2368,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2368.jpg,Carver,91674,None,None,POINT (-10465155.126 5579058.785)
13512,carver_1937_wm-26-2369,None,Carver,1937,wm-26-2369,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2369.jpg,Carver,91675,None,None,POINT (-10465099.911 5580868.601)
13513,carver_1937_wm-26-2370,None,Carver,1937,wm-26-2370,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2370.jpg,Carver,91676,None,None,POINT (-10465044.585 5582678.778)
13514,carver_1937_wm-26-2371,None,Carver,1937,wm-26-2371,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2371.jpg,Carver,91677,None,None,POINT (-10464989.371 5584489.317)
13515,carver_1937_wm-26-2372,None,Carver,1937,wm-26-2372,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1937/wm-26-2372.jpg,Carver,91678,None,None,POINT (-10464934.045 5586300.061)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14466,carver_1963_wm-1dd-111,None,Carver,1963,wm-1dd-111,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-111.jpg,Carver,92355,None,None,POINT (-10428717.028 5574748.276)
14467,carver_1963_wm-1dd-113,None,Carver,1963,wm-1dd-113,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-113.jpg,Carver,92356,None,None,POINT (-10424133.337 5574825.175)
14468,carver_1963_wm-1dd-228,None,Carver,1963,wm-1dd-228,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-228.jpg,Carver,92362,None,None,POINT (-10435676.055 5570228.328)
14469,carver_1963_wm-1dd-230,None,Carver,1963,wm-1dd-230,None,None,None,20000,None,None,None,https://geo.lib.umn.edu/carver/y1963/wm-1dd-230.jpg,Carver,92363,None,None,POINT (-10431122.308 5570238.503)


### Save a subset index table with a fancy name

In [17]:
# Build filename components
county_name = COUNTY.strip().lower().replace(" ", "_")

year_suffix = "_".join(
    f"{year % 100:02d}" for year in sorted(set(YEARS))
)

filename = f"subset_{SCALE}_{county_name}_{year_suffix}.gpkg"

# Create output directory if needed
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / filename

# Prevent accidental overwrites
if output_path.exists():
    raise FileExistsError(f"File already exists: {output_path}")

# Save subset
subset_gdf.to_file(output_path, layer="photos", driver="GPKG")

print(f"Saved {len(subset_gdf):,} records to:")
print(output_path)

Saved 871 records to:
data\historic_airphoto_index\subsets\subset_20000_carver_37_38_40_51_57_63.gpkg


In [16]:
subset_gdf.columns

Index(['cty_year_id', 'status', 'county', 'year', 'photo_id', 'h_extent',
       'gcp_count', 'confidence', 'scale', 'dpi_x', 'width_px',
       'worldfile_json', 'url', 'flight_code', 'source_id', 'date_download',
       'date_georef', 'geometry'],
      dtype='str')